In [ ]:
%stop_session

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

In [ ]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE         = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO       = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE         = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO       = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD                    = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV                = f"{BASE}/gold/gmv_daily_by_subsidiary"

### BRONZE — purchase_extra_info (eventos CDC)

In [ ]:
# Criar DataFrame diretamente (dados do enunciado - Exercício 2)
df_purchase_extra_info_bronze = spark.createDataFrame([
    ("2023-01-23 00:05:00", "2023-01-23", 55, "nacional"),
    ("2023-01-25 23:59:59", "2023-01-25", 56, "internacional"),
    ("2023-02-28 01:10:00", "2023-02-28", 69, "nacional"),
    ("2023-03-12 07:00:00", "2023-03-12", 69, "internacional")
], schema=[
    "transaction_datetime",
    "transaction_date",
    "purchase_id",
    "subsidiary"
])

df_purchase_extra_info_bronze = df_purchase_extra_info_bronze \
    .withColumn("transaction_datetime", col("transaction_datetime").cast("timestamp")) \
    .withColumn("transaction_date", col("transaction_date").cast("date")) \
    .withColumn("purchase_id", col("purchase_id").cast("bigint")) \
    .withColumn("subsidiary", col("subsidiary").cast("string"))

df_purchase_extra_info_bronze = df_purchase_extra_info_bronze.withColumn(
    "ingestion_date",
    to_utc_timestamp(current_timestamp(), "UTC")
)

# Persistir Bronze em DELTA (append-only)
df_purchase_extra_info_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("transaction_date") \
    .save(PATH_BRONZE_EXTRA_INFO)

In [ ]:
# uat
df_purchase_extra_info_bronze.createOrReplaceTempView("vw_purchase_extra_info_bronze")

spark.sql("SELECT * FROM vw_purchase_extra_info_bronze").show(truncate = False)